# Manim Studio — Session 01

Technical visualizations for *Data, Representations, and the First Model* (Manim Community Edition).

Scene ideas inspired by — but re-implemented from scratch, not copied from — Thomas Countz, *Calculate the Decision Boundary of a Single Perceptron*, and Joshua Thompson, *Visualizing the MLP: A Composition of Transformations*.

**Videos vs. stills:** animate only when motion is the message. Two scenes are rendered as **MP4 video** (the perceptron learning, the space warping); two are rendered as **static PNG** via Manim's `-s` (save-last-frame) flag — same design, no separate tool. Run the export cell at the end to copy clean files into `media/session_01/` and print ready-to-paste embed snippets.


In [ ]:
# Setup (run once per kernel). First time only:  pip install manim   (+ ffmpeg)
from manim import *
config.media_dir = 'media'
config.background_color = '#0f172a'
import manim; print('Manim', manim.__version__)


### 1 · Perceptron learns the decision boundary  *(video)*
The learning rule nudges the line after each misclassified point until the classes are separated. The red arrow is the weight vector **w**, orthogonal to the boundary `w·x + b = 0`.

In [ ]:
%%manim -qm -v WARNING PerceptronLearns
from manim import *
import numpy as np

class PerceptronLearns(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        ax = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=6.3, y_length=6.3,
                  axis_config={'include_tip': False, 'stroke_opacity': 0.4})
        rng = np.random.default_rng(2)
        A = rng.normal([1.3, 1.3], 0.45, size=(8, 2))
        B = rng.normal([-1.3, -1.3], 0.45, size=(8, 2))
        X = np.vstack([A, B]); Y = np.array([1]*8 + [0]*8)
        dA = VGroup(*[Dot(ax.c2p(*p), color=BLUE_B, radius=0.08) for p in A])
        dB = VGroup(*[Dot(ax.c2p(*p), color=YELLOW, radius=0.08) for p in B])
        title = Text('Perceptron learning rule', font_size=26).to_edge(UP, buff=0.25)
        caption = Text('boundary:  w . x + b = 0      (w is orthogonal to it)', font_size=20).to_edge(DOWN, buff=0.25)
        self.play(Create(ax), Write(title)); self.play(FadeIn(dA), FadeIn(dB), FadeIn(caption))

        def boundary_line(w, b):
            pts = []
            for x in (-3, 3):
                if abs(w[1]) > 1e-9:
                    yy = -(w[0]*x + b)/w[1]
                    if -3 <= yy <= 3: pts.append((x, yy))
            for yv in (-3, 3):
                if abs(w[0]) > 1e-9:
                    xx = -(w[1]*yv + b)/w[0]
                    if -3 <= xx <= 3: pts.append((xx, yv))
            uniq = []
            for p in pts:
                if all(abs(p[0]-q[0]) + abs(p[1]-q[1]) > 1e-6 for q in uniq): uniq.append(p)
            if len(uniq) < 2: uniq = [(-3, 0), (3, 0)]
            return Line(ax.c2p(*uniq[0]), ax.c2p(*uniq[1]), color=GREEN_B, stroke_width=4)

        def group(w, b):
            line = boundary_line(w, b)
            n = max(np.linalg.norm(w), 1e-9)
            foot = np.clip(-b * np.array(w) / (n*n), -2.6, 2.6)
            arrow = Arrow(ax.c2p(*foot), ax.c2p(*(foot + np.array(w)/n)), color=RED_B,
                          buff=0, stroke_width=4, max_tip_length_to_length_ratio=0.3)
            return VGroup(line, arrow)

        w = np.array([-1.5, 2.0]); b = 0.5; eta = 0.2
        grp = group(w, b); self.play(Create(grp))
        changed, epoch = True, 0
        while changed and epoch < 6:
            changed, epoch = False, epoch + 1
            for xi, yi in zip(X, Y):
                pred = 1 if (w @ xi + b) >= 0 else 0
                if pred != yi:
                    w = w + eta*(yi - pred)*xi; b = b + eta*(yi - pred); changed = True
                    self.play(Transform(grp, group(w, b)), run_time=0.4)
        self.wait(1.5)


### 2 · A hidden layer warps the space  *(video)*
`h = tanh(Wx + b)`: a linear change of basis followed by the tanh nonlinearity bends the input plane until XOR is linearly separable in the latent space.

In [ ]:
%%manim -qm -v WARNING MLPWarp
from manim import *
import numpy as np

class MLPWarp(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        O = np.array([0.0, -0.6, 0.0]); S = 2.2
        def Cc(x, y): return O + np.array([x*S, y*S, 0.0])
        def inv(p): return ((p[0]-O[0])/S, (p[1]-O[1])/S)
        def h(x, y):
            return (np.tanh(1.5*x + 1.5*y - 0.75), np.tanh(1.5*x + 1.5*y - 2.25))
        F = lambda p: Cc(*h(*inv(p)))
        title = Text('Hidden layer  h = tanh(Wx + b)  warps the input space', font_size=23).to_edge(UP, buff=0.25)
        grid = VGroup(); lo, hi = -0.6, 1.6
        vals = np.linspace(lo, hi, 6); samp = np.linspace(lo, hi, 30)
        for x in vals: grid.add(VMobject().set_points_as_corners([Cc(x, t) for t in samp]))
        for y in vals: grid.add(VMobject().set_points_as_corners([Cc(t, y) for t in samp]))
        grid.set_stroke(BLUE_D, width=2, opacity=0.45)
        P = [((0, 0), BLUE_B), ((1, 1), BLUE_B), ((0, 1), YELLOW), ((1, 0), YELLOW)]
        dots = VGroup(*[Dot(Cc(x, y), color=c, radius=0.12) for (x, y), c in P])
        self.play(Write(title), Create(grid), FadeIn(dots)); self.wait(0.5)
        targets = [Cc(*h(x, y)) for (x, y), c in P]
        self.play(grid.animate.apply_function(F),
                  *[d.animate.move_to(t) for d, t in zip(dots, targets)], run_time=3)
        sep = Line(Cc(-0.2, -1.0), Cc(1.0, 0.2), color=GREEN_B, stroke_width=4)
        ok = Text('in the latent space, one straight line separates XOR', font_size=20, color=GREEN_B).to_edge(DOWN, buff=0.3)
        self.play(Create(sep), Write(ok)); self.wait(2)


### 3 · Activation functions and their derivatives  *(static image)*
Rendered with the `-s` flag — Manim saves the final frame as a PNG. Solid = function, dashed = derivative.

In [ ]:
%%manim -s -qh -v WARNING ActivationFunctions
from manim import *
import numpy as np

class ActivationFunctions(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        sig = lambda x: 1/(1 + np.exp(-x))
        funcs = [
            ('Step',       lambda x: 1.0 if x >= 0 else 0.0, None),
            ('Sigmoid',    sig, lambda x: sig(x)*(1 - sig(x))),
            ('Tanh',       lambda x: np.tanh(x), lambda x: 1 - np.tanh(x)**2),
            ('ReLU',       lambda x: max(0.0, x), lambda x: 1.0 if x > 0 else 0.0),
            ('Leaky ReLU', lambda x: x if x > 0 else 0.1*x, lambda x: 1.0 if x > 0 else 0.1),
            ('GELU',       lambda x: x*sig(1.702*x), None),
        ]
        grid = VGroup()
        for name, f, df in funcs:
            ax = Axes(x_range=[-2.5, 2.5, 1], y_range=[-1.3, 2.8, 1], x_length=3.4, y_length=2.4,
                      axis_config={'include_tip': False, 'stroke_opacity': 0.4, 'stroke_width': 2})
            cell = VGroup(ax, ax.plot(f, x_range=[-2.5, 2.5], color=BLUE_B, use_smoothing=False))
            if df is not None:
                cell.add(DashedVMobject(ax.plot(df, x_range=[-2.5, 2.5], color=YELLOW, use_smoothing=False), num_dashes=40))
            cell.add(Text(name, font_size=20).next_to(ax, UP, buff=0.1))
            grid.add(cell)
        grid.arrange_in_grid(rows=2, cols=3, buff=0.7)
        grid.scale_to_fit_width(12.6).to_edge(UP, buff=0.6)
        legend = VGroup(Text('— function', font_size=18, color=BLUE_B),
                        Text('-- derivative', font_size=18, color=YELLOW)).arrange(RIGHT, buff=0.7).to_edge(DOWN, buff=0.25)
        self.add(grid, legend)


### 4 · Standardization conditions the loss surface  *(static image)*
Rendered with `-s`. Elongated contours (slow descent) become round after standardizing to unit variance.

In [ ]:
%%manim -s -qh -v WARNING Standardization
from manim import *
import numpy as np

class Standardization(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(0)
        pts = rng.normal(0, 1, size=(60, 2)) * np.array([2.2, 0.5])
        axL = Axes(x_range=[-6, 6, 2], y_range=[-3, 3, 1], x_length=6, y_length=4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(LEFT, buff=0.5).shift(DOWN*0.2)
        cloud = VGroup(*[Dot(axL.c2p(*p), color=BLUE_B, radius=0.05) for p in pts])
        ell = VGroup(*[Ellipse(width=w*2.0, height=w*0.7, color=YELLOW, stroke_opacity=0.5).move_to(axL.c2p(0, 0)) for w in (1, 2, 3)])
        tl = Text('raw: different scales', font_size=20).next_to(axL, UP, buff=0.15)
        axR = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=4.4, y_length=4.4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(RIGHT, buff=0.5).shift(DOWN*0.2)
        z = (pts - pts.mean(0)) / pts.std(0)
        cloudZ = VGroup(*[Dot(axR.c2p(*p), color=BLUE_B, radius=0.05) for p in z])
        circ = VGroup(*[Circle(radius=r, color=GREEN_B, stroke_opacity=0.6).move_to(axR.c2p(0, 0)) for r in (0.8, 1.6, 2.4)])
        tr = Text('standardized: unit variance', font_size=18).next_to(axR, UP, buff=0.15)
        cap = Text('round contours -> faster gradient descent', font_size=20, color=GREEN_B).to_edge(DOWN, buff=0.25)
        self.add(axL, cloud, ell, tl, axR, cloudZ, circ, tr, cap)


### 5 · MLP architecture and parameter count  *(static image)*
The Go2 controller 48 → 256 → 256 → 12 as a fully-connected network, with per-layer parameter counts. Rendered with `-s`.

In [ ]:
%%manim -s -qh -v WARNING MLPArchitecture
from manim import *
import numpy as np

class MLPArchitecture(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        layers = [(-5.0, 5, 'Input', '48'), (-1.7, 6, 'Hidden 1', '256 . ReLU'),
                  (1.6, 6, 'Hidden 2', '256 . ReLU'), (4.9, 4, 'Output', '12')]
        dots = [True, True, True, False]
        sp = 0.5
        def ys(n): return [(n-1)/2*sp - i*sp for i in range(n)]
        conns = VGroup()
        for (x0, n0, _, _), (x1, n1, _, _) in zip(layers[:-1], layers[1:]):
            for y0 in ys(n0):
                for y1 in ys(n1):
                    conns.add(Line([x0, y0, 0], [x1, y1, 0], color='#3a4a63', stroke_width=1, stroke_opacity=0.5))
        self.add(conns)
        for (x, n, name, dim), d in zip(layers, dots):
            for y in ys(n):
                self.add(Dot([x, y, 0], radius=0.13, color=BLUE_B))
            if d:
                base = ys(n)[-1] - 0.32
                self.add(VGroup(*[Dot([x, base - 0.12*k, 0], radius=0.025, color='#e2e8f0') for k in range(3)]))
            self.add(Text(name, font_size=22).move_to([x, -2.5, 0]))
            self.add(Text(dim, font_size=18, color='#9cc2f5').move_to([x, -2.95, 0]))
        params = [('256x48 + 256', '= 12,544', -3.35), ('256x256 + 256', '= 65,792', -0.05), ('12x256 + 12', '= 3,084', 3.25)]
        for l1, l2, mx in params:
            self.add(VGroup(Text(l1, font_size=18, color=GREEN_B), Text(l2, font_size=18, color=GREEN_B)).arrange(DOWN, buff=0.08).move_to([mx, 2.6, 0]))
        self.add(Text('Total: 81,420 trainable parameters', font_size=22).move_to([0, -3.5, 0]))
        self.add(Text('MLP motor controller for the Go2', font_size=26).to_edge(UP, buff=0.3))


### Export & embed
Render the two videos with `-qh` for the final 1080p versions if you like, then run this cell. It copies the latest MP4s **and** the `-s` PNG stills into `media/session_01/` and prints ready-to-paste embed blocks for `sessions/session_01.md`.

In [ ]:
import glob, os, shutil

VIDEOS = {'PerceptronLearns': 'perceptron_boundary', 'MLPWarp': 'mlp_warp'}
IMAGES = {'ActivationFunctions': 'activation_functions', 'Standardization': 'standardization', 'MLPArchitecture': 'mlp_architecture'}
os.makedirs('media/session_01', exist_ok=True)

def newest(pattern):
    hits = glob.glob(pattern, recursive=True)
    return max(hits, key=os.path.getmtime) if hits else None

for scene, name in VIDEOS.items():
    src = newest(f'media/videos/**/{scene}.mp4')
    if not src: print(f'(no video yet for {scene})'); continue
    dst = f'media/session_01/{name}.mp4'; shutil.copy(src, dst); print('saved', dst)
    print(f'  <video src="media/session_01/{name}.mp4" controls loop muted playsinline style="max-width:100%;border-radius:10px;"></video>')

for scene, name in IMAGES.items():
    src = newest(f'media/images/**/{scene}*.png')
    if not src: print(f'(no still yet for {scene} — run its cell with the -s flag)'); continue
    dst = f'media/session_01/{name}.png'; shutil.copy(src, dst); print('saved', dst)
    print(f'  <img src="media/session_01/{name}.png" style="max-width:100%;border-radius:10px;">')
